![](https://mcd.unison.mx/wp-content/themes/awaken/img/logo_mcd.png)

# Tensores y diferenciación automática en pyTorch

## Aprendizaje Automático Aplicado

### Maestría en Ciencia de Datos

#### **Julio Waissman**, 2026

[**Abrir en google Colab**](https://colab.research.google.com/github/mcd-unison/aaa-curso/blob/main/ejemplos/tensor_tutorial.ipynb)

Extraido y adaptado de [los tutoriales de pyTorch](https://pytorch.org/tutorials/beginner/basics/index.html).



## Tensores

Los tensores son una estructura de datos especializada muy similar a los
arreglos y matrices. En PyTorch, usamos tensores para codificar las
entradas y salidas de un modelo, así como los parámetros del modelo.

Los tensores son similares a los [nd-arrays de NumPy](https://numpy.org/),
con la diferencia de que los tensores pueden ejecutarse en GPUs u otros
aceleradores de hardware. De hecho, los tensores y los arreglos de NumPy
pueden compartir la misma memoria subyacente, eliminando la necesidad de
copiar datos.

Los tensores también están optimizados para la diferenciación automática. Si estás familiarizado con los nd-arrays, te sentirás como en casa con la API de Tensor. Si no, ¡sigue el tutorial!


In [1]:
import torch
import numpy as np

## Inicialización de un Tensor

Los tensores se pueden inicializar de diversas formas. Observa los
siguientes ejemplos:

**Directamente desde datos**

Los tensores se pueden crear directamente desde datos. El tipo de dato se
infiere automáticamente.


In [2]:
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)

**Desde un arreglo de NumPy**

Los tensores se pueden crear desde arreglos de NumPy (y viceversa).


In [3]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

**Desde otro tensor:**

El nuevo tensor conserva las propiedades (forma, tipo de dato) del tensor
argumento, a menos que se anulen explícitamente.


In [4]:
x_ones = torch.ones_like(x_data) # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # overrides the datatype of x_data
print(f"Random Tensor: \n {x_rand} \n")

Ones Tensor: 
 tensor([[1, 1],
        [1, 1]]) 

Random Tensor: 
 tensor([[0.2796, 0.0101],
        [0.5949, 0.5909]]) 



**Con valores aleatorios o constantes:**

`shape` es una tupla de dimensiones del tensor. En las funciones de
abajo, determina la dimensionalidad del tensor de salida.


In [5]:
shape = (2,3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.4813, 0.0805, 0.4575],
        [0.6477, 0.2005, 0.5970]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


## Atributos de un Tensor

Los atributos de un tensor describen su forma, tipo de dato y el
dispositivo en el que se almacena.


In [6]:
tensor = torch.rand(3,4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


## Operaciones con Tensores

Más de 1200 operaciones con tensores, incluyendo aritmética, álgebra
lineal, manipulación de matrices (transposición, indexación, slicing),
muestreo y más, están descritas exhaustivamente
[aquí](https://pytorch.org/docs/stable/torch.html).

Cada una de estas operaciones puede ejecutarse en la CPU y en un
[Acelerador](https://pytorch.org/docs/stable/torch.html#accelerators)
como CUDA, MPS, MTIA o XPU. Si usas Colab, asigna un acelerador yendo a
Entorno de ejecución \> Cambiar tipo de entorno de ejecución \> GPU.

Por defecto, los tensores se crean en la CPU. Necesitamos mover
explícitamente los tensores al acelerador usando el método `.to` (después
de verificar la disponibilidad del acelerador). Ten en cuenta que copiar
tensores grandes entre dispositivos puede ser costoso en tiempo y memoria.


In [7]:
# We move our tensor to the current accelerator if available
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())

Prueba algunas de las operaciones de la lista. Si estás familiarizado con
la API de NumPy, encontrarás que la API de Tensor es muy sencilla de usar.


**Indexación y slicing estándar al estilo NumPy:**


In [8]:
tensor = torch.ones(4, 4)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[..., -1]}")
tensor[:,1] = 0
print(tensor)

First row: tensor([1., 1., 1., 1.])
First column: tensor([1., 1., 1., 1.])
Last column: tensor([1., 1., 1., 1.])
tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


**Unión de tensores** Puedes usar `torch.cat` para concatenar una
secuencia de tensores a lo largo de una dimensión dada. Consulta también
[torch.stack](https://pytorch.org/docs/stable/generated/torch.stack.html),
otro operador de unión de tensores que difiere sutilmente de `torch.cat`.


In [9]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)

tensor([[1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.]])


**Operaciones aritméticas**


In [10]:
# This computes the matrix multiplication between two tensors. y1, y2, y3 will have the same value
# ``tensor.T`` returns the transpose of a tensor
y1 = tensor @ tensor.T
y2 = tensor.matmul(tensor.T)

y3 = torch.rand_like(y1)
torch.matmul(tensor, tensor.T, out=y3)


# This computes the element-wise product. z1, z2, z3 will have the same value
z1 = tensor * tensor
z2 = tensor.mul(tensor)

z3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=z3)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])

**Tensores de un solo elemento** Si tienes un tensor de un único elemento,
por ejemplo al agregar todos los valores de un tensor en uno solo, puedes
convertirlo a un valor numérico de Python usando `item()`:


In [11]:
agg = tensor.sum()
agg_item = agg.item()
print(agg_item, type(agg_item))

12.0 <class 'float'>


**Operaciones in-place** Las operaciones que almacenan el resultado en el
operando se denominan in-place. Se identifican con el sufijo `_`. Por
ejemplo: `x.copy_(y)`, `x.t_()`, modificarán `x`.


In [12]:
print(f"{tensor} \n")
tensor.add_(5)
print(tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor([[6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.]])


Las operaciones in-place ahorran algo de memoria, pero pueden ser problemáticas al calcular derivadas debido a la pérdida inmediata del historial. Por ello, se desaconseja su uso.

## Puente con NumPy 

Los tensores en la CPU y los arreglos de NumPy pueden compartir sus
ubicaciones de memoria subyacentes, de modo que cambiar uno cambiará el
otro.


**De Tensor a arreglo NumPy**

In [13]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]


Un cambio en el tensor se refleja en el arreglo de NumPy.


In [14]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


**De arreglo NumPy a Tensor**

In [15]:
n = np.ones(5)
t = torch.from_numpy(n)

Los cambios en el arreglo de NumPy se reflejan en el tensor.


In [16]:
np.add(n, 1, out=n)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.], dtype=torch.float64)
n: [2. 2. 2. 2. 2.]


## Diferenciación automática con `torch.autograd`

Al entrenar redes neuronales, el algoritmo más utilizado es la
**retropropagación** (*back propagation*). En este algoritmo, los
parámetros (pesos del modelo) se ajustan según el **gradiente** de la
función de pérdida con respecto al parámetro dado.

Para calcular esos gradientes, PyTorch cuenta con un motor de
diferenciación incorporado llamado `torch.autograd`. Este soporta el
cálculo automático del gradiente para cualquier grafo computacional.

Considera la red neuronal más simple de una capa, con entrada `x`,
parámetros `w` y `b`, y alguna función de pérdida. Se puede definir en
PyTorch de la siguiente manera:

In [17]:
import torch

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

## Tensores, Funciones y Grafo Computacional

Este código define el siguiente **grafo computacional**:

![](https://pytorch.org/tutorials/_static/img/basics/comp-graph.png)

En esta red, `w` y `b` son **parámetros** que necesitamos optimizar. Por
tanto, necesitamos poder calcular los gradientes de la función de pérdida
con respecto a esas variables. Para ello, establecemos la propiedad
`requires_grad` de esos tensores.

Puedes establecer el valor de `requires_grad` al crear un tensor, o más tarde usando el método `x.requires_grad_(True)`.

Una función que aplicamos a los tensores para construir el grafo
computacional es en realidad un objeto de la clase `Function`. Este
objeto sabe cómo calcular la función en la dirección *forward*, y también
cómo calcular su derivada durante el paso de *retropropagación*. Una
referencia a la función de retropropagación se almacena en la propiedad
`grad_fn` de un tensor. Puedes encontrar más información sobre `Function`
[en la documentación](https://pytorch.org/docs/stable/autograd.html#function).

Solo podemos obtener las propiedades `grad` para los nodos hoja del grafo computacional, que tienen la propiedad `requires_grad` establecida en `True`. Para todos los demás nodos del grafo, los gradientes no estarán disponibles.

Solo podemos realizar cálculos de gradiente usando `backward` una vez sobre un grafo dado, por razones de rendimiento. Si necesitamos hacer varias llamadas a `backward` sobre el mismo grafo, debemos pasar `retain_graph=True` a la llamada de `backward`.

In [18]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x7fdbd445b610>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x7fdbd3db1c90>


## Desactivar el Seguimiento de Gradientes

Por defecto, todos los tensores con `requires_grad=True` rastrean su
historial computacional y soportan el cálculo de gradientes. Sin embargo,
hay casos en los que no necesitamos hacer eso; por ejemplo, cuando hemos
entrenado el modelo y solo queremos aplicarlo a datos de entrada, es decir,
solo queremos hacer computaciones *forward* a través de la red. Podemos
detener el seguimiento de computaciones envolviendo nuestro código con un
bloque `torch.no_grad()`:

In [19]:
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

True
False


Otra forma de lograr el mismo resultado es usar el método `detach()` sobre
el tensor:

In [20]:
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

False


Hay razones por las que podrías querer desactivar el seguimiento de gradientes:

-   Para marcar algunos parámetros de tu red neuronal como **parámetros
    congelados**.
-   Para **acelerar los cálculos** cuando solo realizas el paso forward,
    ya que las operaciones sobre tensores que no rastrean gradientes son
    más eficientes.

## Más sobre Grafos Computacionales

Conceptualmente, autograd mantiene un registro de los datos (tensores) y
de todas las operaciones ejecutadas (junto con los nuevos tensores
resultantes) en un grafo acíclico dirigido (DAG, por sus siglas en inglés)
compuesto por objetos
[Function](https://pytorch.org/docs/stable/autograd.html#torch.autograd.Function).
En este DAG, las hojas son los tensores de entrada y las raíces son los
tensores de salida. Al recorrer este grafo desde las raíces hasta las
hojas, los gradientes se pueden calcular automáticamente usando la regla
de la cadena.

En un paso forward, autograd hace dos cosas simultáneamente:

-   ejecuta la operación solicitada para calcular el tensor resultante,
-   mantiene la *función de gradiente* de la operación en el DAG.

El paso backward se inicia cuando se llama a `.backward()` sobre la raíz
del DAG. `autograd` entonces:

-   calcula los gradientes a partir de cada `.grad_fn`,
-   los acumula en el atributo `.grad` del tensor correspondiente,
-   usando la regla de la cadena, los propaga hasta los tensores hoja.

Un aspecto importante a tener en cuenta es que el grafo se recrea desde cero; después de cada llamada a `.backward()`, autograd comienza a poblar un nuevo grafo. Esto es precisamente lo que te permite usar sentencias de control de flujo en tu modelo; puedes cambiar la forma, el tamaño y las operaciones en cada iteración si es necesario.